In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1363")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
-- Tag email and phone columns in gold.dim_customer as PII
SET TAG ON COLUMN fk_ce_2026_07_1363.gold.dim_customer.email pii = true;
SET TAG ON COLUMN fk_ce_2026_07_1363.gold.dim_customer.email pii_type = email;
SET TAG ON COLUMN fk_ce_2026_07_1363.gold.dim_customer.phone pii = true;
SET TAG ON COLUMN fk_ce_2026_07_1363.gold.dim_customer.phone pii_type = phone;


In [0]:
%sql
-- Create UDF to mask email (first character, then ***@domain)
CREATE OR REPLACE FUNCTION fk_ce_2026_07_1363.gold.mask_email(email STRING)
RETURNS STRING
RETURN CASE 
  WHEN email IS NULL THEN NULL
  ELSE 
    CONCAT(
      COALESCE(SUBSTRING(email, 1, 1), ''),
      '***@',
      REGEXP_EXTRACT(email, '@(.+)$')
    )
END;

-- Create UDF to mask phone (XXXXXX + last 4 digits)
CREATE OR REPLACE FUNCTION fk_ce_2026_07_1363.gold.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE 
  WHEN phone IS NULL OR LENGTH(phone) < 4 THEN NULL
  ELSE CONCAT('XXXXXX', SUBSTRING(phone, LENGTH(phone) - 3, 4))
END;


In [0]:
# %sql
# -- Create column mask policies for dim_customer.email and dim_customer.phone
# CREATE POLICY mask_email_policy
# ON TABLE fk_ce_2026_07_1363.gold.dim_customer
# COMMENT 'Mask emails for non-admin and users without can_view_pii'
# COLUMN MASK fk_ce_2026_07_1363.gold.mask_email
# TO `users` EXCEPT `fk_admins`
# FOR TABLES
# WHEN NOT (entitlement('can_view_pii') = 'TRUE')
# MATCH COLUMNS has_tag_value('pii', 'email') AS email
# ON COLUMN email;

# CREATE POLICY mask_phone_policy
# ON TABLE fk_ce_2026_07_1363.gold.dim_customer
# COMMENT 'Mask phones for non-admin and users without can_view_pii'
# COLUMN MASK fk_ce_2026_07_1363.gold.mask_phone
# TO `users` EXCEPT `fk_admins`
# FOR TABLES
# WHEN NOT (entitlement('can_view_pii') = 'TRUE')
# MATCH COLUMNS has_tag_value('pii', 'phone') AS phone
# ON COLUMN phone;


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.dim_date AS
SELECT 
    CAST(DATE_FORMAT(date_val, 'yyyyMMdd') AS INT) AS date_key,
    date_val AS calendar_date,
    DATE_FORMAT(date_val, 'EEEE') AS day_name,
    WEEKOFYEAR(date_val) AS iso_week,
    CASE WHEN DAYOFWEEK(date_val) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT EXPLODE(SEQUENCE(DATE '2026-03-01', DATE '2026-03-31', INTERVAL 1 DAY)) AS date_val
);

In [0]:
%sql
select * from fk_ce_2026_07_1363.gold.dim_date

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.dim_store AS
SELECT * FROM IDENTIFIER(:catalog).silver.stores;


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.dim_product AS
SELECT * EXCEPT(tags, last_changed_snapshot_date)
FROM IDENTIFIER(:catalog).silver.products_type1;


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.dim_customer AS
SELECT monotonically_increasing_id() AS customer_sk, *
FROM IDENTIFIER(:catalog).silver.customers_type2;


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.fact_orders AS
SELECT
  o.order_id,
  CAST(DATE_FORMAT(o.order_ts, 'yyyyMMdd') AS INT) AS order_date_key,
  c.customer_sk,
  o.customer_id,
  o.store_id,
  o.order_status,
  o.payment_method,
  o.channel,
  COUNT(oi.line_number) AS line_count,
  SUM(oi.quantity) AS units,
  SUM(oi.gross_amount) AS gross_amount,
  SUM(oi.discount_amount) AS discount_amount,
  SUM(oi.net_amount) AS net_item_amount,
  0 AS delivery_fee,
  SUM(oi.net_amount) + 0 AS order_total,
  o.promised_mins,
  o.actual_mins,
  CASE WHEN o.actual_mins IS NOT NULL AND o.promised_mins IS NOT NULL
    THEN o.actual_mins <= o.promised_mins ELSE NULL END AS is_on_time
FROM IDENTIFIER(:catalog).silver.orders o
LEFT JOIN IDENTIFIER(:catalog).silver.order_items oi ON o.order_id = oi.order_id
LEFT JOIN IDENTIFIER(:catalog).gold.dim_customer c ON o.customer_id = c.customer_id
GROUP BY o.order_id, o.order_ts, c.customer_sk, o.customer_id, o.store_id, o.order_status, o.payment_method, o.channel, o.promised_mins, o.actual_mins
;

In [0]:
%sql
select * from fk_ce_2026_07_1363.gold.fact_orders

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog).gold.agg_daily_store_sales AS
SELECT
  dd.calendar_date AS order_date,
  s.store_id,
  s.city,
  COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0) AS delivered_orders,
  COALESCE(SUM(CASE WHEN f.order_status = 'CANCELLED' THEN 1 ELSE 0 END), 0) AS cancelled_orders,
  COALESCE(SUM(f.units), 0) AS units_sold,
  COALESCE(SUM(f.gross_amount), 0) AS gross_amount,
  COALESCE(SUM(f.discount_amount), 0) AS discount_amount,
  COALESCE(SUM(f.net_item_amount), 0) AS net_item_revenue,
  COALESCE(SUM(f.delivery_fee), 0) AS delivery_fee_revenue,
  COALESCE(SUM(CASE WHEN f.is_on_time THEN 1 ELSE 0 END), 0) AS on_time_orders,
  CASE WHEN COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0) = 0
    THEN NULL
    ELSE ROUND(
      100.0 * COALESCE(SUM(CASE WHEN f.is_on_time THEN 1 ELSE 0 END), 0) 
      / COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0), 2)
  END AS on_time_rate_pct
FROM IDENTIFIER(:catalog).gold.dim_date dd
CROSS JOIN IDENTIFIER(:catalog).gold.dim_store s
LEFT JOIN IDENTIFIER(:catalog).gold.fact_orders f
  ON f.order_date_key = CAST(DATE_FORMAT(dd.calendar_date,'yyyyMMdd') AS INT)
  AND f.store_id = s.store_id
GROUP BY dd.calendar_date, s.store_id, s.city
ORDER BY dd.calendar_date, s.store_id;


In [0]:
%sql
select * from fk_ce_2026_07_1363.gold.agg_daily_store_sales;


In [0]:
%sql
CREATE OR REPLACE VIEW IDENTIFIER(:catalog).gold.v_kpi_summary AS
SELECT
  SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
  SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
  SUM(net_item_amount) AS net_item_revenue,
  ROUND(SUM(net_item_amount) / NULLIF(SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0), 2) AS avg_order_value,
  ROUND(100.0 * SUM(CASE WHEN is_on_time AND order_status = 'DELIVERED' THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0), 2) AS on_time_rate_pct,
  ROUND(100.0 * SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) / NULLIF(SUM(CASE WHEN order_status IN ('DELIVERED', 'CANCELLED') THEN 1 ELSE 0 END), 0), 2) AS cancellation_rate_pct,
  COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END) AS active_customers,
  ROUND(100.0 * COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END) FILTER (WHERE delivered_order_count >= 2) / NULLIF(COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END), 0), 2) AS repeat_customer_rate_pct
FROM (
  SELECT *,
    COUNT(*) FILTER (WHERE order_status = 'DELIVERED') OVER (PARTITION BY customer_id) AS delivered_order_count
  FROM IDENTIFIER(:catalog).gold.fact_orders
);
